Live-record power usage data.
ref [pynq-documentation](https://pynq.readthedocs.io/en/v2.3/pynq_package/pynq.pmbus.html) and [Alveo-PYNQ example](https://github.com/Xilinx/Alveo-PYNQ/blob/master/pynq_alveo_examples/notebooks/3_advanced_features/3-live-power-plotting.ipynb)

It seems the Kria does not support it by default. Does the bitfile need to be generated with some special configurations?

In [ ]:
from pynq import Device

sensors = Device.active_device.sensors
sensors
rails = Rail.get_rails()
rails

{'0v85': XrtRail {name=0v85, voltage=Sensor {name=0v85_vol, value=0.0V}},
 '12v_aux': XrtRail {name=12v_aux, voltage=Sensor {name=12v_aux_vol, value=0.0V}, current=Sensor {name=12v_aux_curr, value=0.0A}, power=Sensor {name=12v_aux_power, value=0.0W}},
 '12v_pex': XrtRail {name=12v_pex, voltage=Sensor {name=12v_pex_vol, value=0.0V}, current=Sensor {name=12v_pex_curr, value=0.0A}, power=Sensor {name=12v_pex_power, value=0.0W}},
 '12v_sw': XrtRail {name=12v_sw, voltage=Sensor {name=12v_sw_vol, value=0.0V}},
 '1v8': XrtRail {name=1v8, voltage=Sensor {name=1v8_vol, value=0.0V}},
 '3v3_aux': XrtRail {name=3v3_aux, voltage=Sensor {name=3v3_aux_vol, value=0.0V}},
 '3v3_pex': XrtRail {name=3v3_pex, voltage=Sensor {name=3v3_pex_vol, value=0.0V}},
 'mgt0v9avcc': XrtRail {name=mgt0v9avcc, voltage=Sensor {name=mgt0v9avcc_vol, value=0.0V}},
 'mgtavtt': XrtRail {name=mgtavtt, voltage=Sensor {name=mgtavtt_vol, value=0.0V}},
 'sys_5v5': XrtRail {name=sys_5v5, voltage=Sensor {name=sys_5v5_vol, value=0.0

In [ ]:
from pynq.pmbus import DataRecorder, Rail

# Discover rails using the PYNQ Rail API (fallback to Device sensors if needed)
try:
    rails = Rail.get_rails()
except Exception:
    rails = sensors

# Build a structured map of all available sensor channels
sensor_channels = {}
for rail_name, rail in rails.items():
    rail_channels = {}
    for attr in ("voltage", "current", "power", "temperature"):
        channel = getattr(rail, attr, None)
        if channel is not None:
            rail_channels[attr] = channel
    if rail_channels:
        sensor_channels[rail_name] = rail_channels

# Flatten all channels for DataRecorder
all_channels = []
for rail_channels in sensor_channels.values():
    all_channels.extend(rail_channels.values())

recorder = DataRecorder(*all_channels)

import pandas as pd
f = recorder.frame
recorder.record(0.1)
sensor_channels

In [ ]:
# Check
f.head()

,Mark,12v_aux_power,12v_pex_power,vccint_power
2026-06-02 10:36:14.086582,0.0,0.0,0.0,0.0
2026-06-02 10:36:14.194803,0.0,0.0,0.0,0.0
2026-06-02 10:36:14.303180,0.0,0.0,0.0,0.0
2026-06-02 10:36:14.409416,0.0,0.0,0.0,0.0
2026-06-02 10:36:14.515519,0.0,0.0,0.0,0.0


In [ ]:
import plotly.graph_objs as go

def update_data(frame, start, end, plot):
    ranged = frame[start:end]
    average_ranged = frame[start-pd.tseries.offsets.Second(5):end]
    rolling = (average_ranged['12v_aux_power'] + average_ranged['12v_pex_power']).rolling(
        pd.tseries.offsets.Second(5)
    ).mean()[ranged.index]
    powers = pd.DataFrame(index=ranged.index)
    powers['board_power'] = ranged['12v_aux_power'] + ranged['12v_pex_power']
    powers['rolling'] = rolling
    data = [
        go.Scatter(x=powers.index, y=powers['board_power'], name="Board Power"),
        go.Scatter(x=powers.index, y=powers['rolling'], name="5 Second Avg")
    ]
    plot.update(data=data)



layout = {
    'xaxis': {
        'title': 'Time (s)'
    },
    'yaxis': {
        'title': 'Power (W)',
        'rangemode': 'tozero',
        'autorange': True
    }
}

plot = go.FigureWidget(layout=layout)
plot


In [ ]:
import threading
import time

do_update = True

def thread_func():
    while do_update:
        now = pd.Timestamp.fromtimestamp(time.time())
        past = now - pd.tseries.offsets.Second(60)
        update_data(recorder.frame, past, now, plot)
        time.sleep(0.5)

from threading import Thread
t = Thread(target=thread_func)
t.start()


In [ ]:
# CLOSE thread and clean up

do_update = False
t.join()
recorder.stop()



{'0v85': XrtRail {name=0v85, voltage=Sensor {name=0v85_vol, value=0.0V}},
 '12v_aux': XrtRail {name=12v_aux, voltage=Sensor {name=12v_aux_vol, value=0.0V}, current=Sensor {name=12v_aux_curr, value=0.0A}, power=Sensor {name=12v_aux_power, value=0.0W}},
 '12v_pex': XrtRail {name=12v_pex, voltage=Sensor {name=12v_pex_vol, value=0.0V}, current=Sensor {name=12v_pex_curr, value=0.0A}, power=Sensor {name=12v_pex_power, value=0.0W}},
 '12v_sw': XrtRail {name=12v_sw, voltage=Sensor {name=12v_sw_vol, value=0.0V}},
 '1v8': XrtRail {name=1v8, voltage=Sensor {name=1v8_vol, value=0.0V}},
 '3v3_aux': XrtRail {name=3v3_aux, voltage=Sensor {name=3v3_aux_vol, value=0.0V}},
 '3v3_pex': XrtRail {name=3v3_pex, voltage=Sensor {name=3v3_pex_vol, value=0.0V}},
 'mgt0v9avcc': XrtRail {name=mgt0v9avcc, voltage=Sensor {name=mgt0v9avcc_vol, value=0.0V}},
 'mgtavtt': XrtRail {name=mgtavtt, voltage=Sensor {name=mgtavtt_vol, value=0.0V}},
 'sys_5v5': XrtRail {name=sys_5v5, voltage=Sensor {name=sys_5v5_vol, value=0.0V}},
 'vccint': XrtRail {name=vccint, voltage=Sensor {name=vccint_vol, value=0.0V}, current=Sensor {name=vccint_curr, value=0.0A}, power=Sensor {name=vccint_power, value=0.0W}}}